<a href="https://colab.research.google.com/github/ryukf333/remote-sensing-crop-yield-ml/blob/main/02cdl_mask.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import geemap
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-crop-yield-2026')

print("Drive mounted and Earth Engine connected!")

Mounted at /content/drive
Drive mounted and Earth Engine connected!


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
# Load the TIGER county boundaries feature collection
counties_ee = ee.FeatureCollection("TIGER/2018/Counties")

# Filter for Iowa using its State FIPS code ('19')
iowa_counties_ee = counties_ee.filter(ee.Filter.eq("STATEFP", "19"))

ee_county_count = iowa_counties_ee.size().getInfo()
print(f"Number of Iowa counties in Earth Engine: {ee_county_count}")

Number of Iowa counties in Earth Engine: 99


In [ ]:
import pandas as pd

# Set your project directory (adjust folder name if different)
PROJECT_DIR = '/content/drive/MyDrive/Project_2_Crop_Yield'
PROCESSED_DIR = f"{PROJECT_DIR}/data_processed"
CLEAN_YIELD_FILE = f"{PROCESSED_DIR}/nass_iowa_corn_yield_2019_2024_clean.csv"

# 2. Reload the cleaned yield table
# Note: dtype={'county_fips': str} ensures leading zeros are never lost
yield_clean = pd.read_csv(CLEAN_YIELD_FILE, dtype={"county_fips": str})
print(f"Loaded {len(yield_clean)} rows from {CLEAN_YIELD_FILE}")

# 3. Authenticate and initialize Google Earth Engine
EE_PROJECT_ID = 'ee-crop-yield-2026'
ee.Authenticate()
ee.Initialize(project=EE_PROJECT_ID)  # Add project='your-cloud-project-id' if prompted

# 4. Query US Census TIGER County boundaries in Earth Engine
counties_ee = ee.FeatureCollection("TIGER/2018/Counties")

# Filter for Iowa (State FIPS = 19)
iowa_counties_ee = counties_ee.filter(ee.Filter.eq("STATEFP", "19"))

# Get county metadata from Earth Engine
ee_features = iowa_counties_ee.getInfo()["features"]
ee_fips_list = sorted([feat["properties"]["GEOID"] for feat in ee_features])

# Extract unique county FIPS from your cleaned NASS dataset
nass_fips_list = sorted(yield_clean["county_fips"].unique().tolist())

# 5. Verification checks
print("\n================ COUNTY MATCH AUDIT ================")
print(f"Total Iowa counties in Earth Engine: {len(ee_fips_list)}")
print(f"Unique counties in cleaned NASS data: {len(nass_fips_list)}")

missing_in_ee = set(nass_fips_list) - set(ee_fips_list)
missing_in_nass = set(ee_fips_list) - set(nass_fips_list)

if not missing_in_ee and not missing_in_nass:
    print("\nSUCCESS: All 99 Iowa counties match Earth Engine boundaries perfectly.")
else:
    if missing_in_ee:
        print(f"WARNING: Present in NASS but missing in Earth Engine: {sorted(missing_in_ee)}")
    if missing_in_nass:
        print(f"INFO: Present in Earth Engine but missing in NASS: {sorted(missing_in_nass)}")

Loaded 524 rows from /content/drive/MyDrive/Project_2_Crop_Yield/data_processed/nass_iowa_corn_yield_2019_2024_clean.csv

================ COUNTY MATCH AUDIT ================
Total Iowa counties in Earth Engine: 99
Unique counties in cleaned NASS data: 99

SUCCESS: All 99 Iowa counties match Earth Engine boundaries perfectly.


In [ ]:
# Define the study years
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
cdl_results = []

print("Extracting county-level corn area from Earth Engine CDL...")

for year in YEARS:
    print(f"Processing {year}...")

    # 1. Load the CDL image for the specific year
    cdl = ee.ImageCollection('USDA/NASS/CDL') \
            .filter(ee.Filter.date(f'{year}-01-01', f'{year}-12-31')) \
            .first() \
            .select('cropland')

    # 2. Create a binary corn mask (Corn class code = 1)
    corn_mask = cdl.eq(1)

    # 3. Calculate area in square kilometers
    # Multiply the binary mask (1 or 0) by pixel area in m^2, then convert to km^2
    corn_area_km2 = corn_mask.multiply(ee.Image.pixelArea()).divide(1e6)

    # 4. Aggregate the corn area within each Iowa county
    # tileScale=4 forces Earth Engine to use more memory-efficient chunking
    county_corn_stats = corn_area_km2.reduceRegions(
        collection=iowa_counties_ee,
        reducer=ee.Reducer.sum(),
        scale=30,
        tileScale=4
    )

    # 5. Bring the computed statistics to the Colab environment
    # We select only the needed properties to keep the network payload light
    extracted = county_corn_stats.select(['GEOID', 'sum']).getInfo()

    for feat in extracted['features']:
        cdl_results.append({
            'county_fips': feat['properties']['GEOID'],
            'year': year,
            'corn_area_km2': feat['properties'].get('sum', 0) # Defaults to 0 if no corn found
        })

# 6. Convert to DataFrame and inspect
cdl_df = pd.DataFrame(cdl_results)

# Sort to match the structure of our target table
cdl_df = cdl_df.sort_values(["county_fips", "year"]).reset_index(drop=True)

print("\nExtraction complete! Data preview:")
display(cdl_df.head())

# 7. Save the CDL summary to Drive
CDL_OUTPUT_FILE = f"{PROCESSED_DIR}/cdl_iowa_corn_area_2019_2024.csv"
cdl_df.to_csv(CDL_OUTPUT_FILE, index=False)
print(f"\nCounty-level CDL summary exported successfully to: {CDL_OUTPUT_FILE}")

Extracting county-level corn area from Earth Engine CDL...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...

Extraction complete! Data preview:


,county_fips,year,corn_area_km2
0,19001,2019,482.932853
1,19001,2020,470.617045
2,19001,2021,436.880502
3,19001,2022,435.481235
4,19001,2023,432.559493



County-level CDL summary exported successfully to: /content/drive/MyDrive/Project_2_Crop_Yield/data_processed/cdl_iowa_corn_area_2019_2024.csv


In [ ]:


print("Shape:", cdl_df.shape)

print("\nYears:")
print(sorted(cdl_df["year"].unique()))

print("\nUnique counties:")
print(cdl_df["county_fips"].nunique())

print("\nRecords by year:")
print(
    cdl_df.groupby("year")["county_fips"]
    .nunique()
    .sort_index()
)

print("\nDuplicate county-year records:")
print(
    cdl_df.duplicated(
        subset=["county_fips", "year"]
    ).sum()
)

print("\nMissing values:")
print(cdl_df.isna().sum())

print("\nCorn area summary:")
print(cdl_df["corn_area_km2"].describe())

display(
    cdl_df.sort_values(
        ["county_fips", "year"]
    ).head(5)
)

Shape: (594, 3)

Years:
[np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Unique counties:
99

Records by year:
year
2019    99
2020    99
2021    99
2022    99
2023    99
2024    99
Name: county_fips, dtype: int64

Duplicate county-year records:
0

Missing values:
county_fips      0
year             0
corn_area_km2    0
dtype: int64

Corn area summary:
count     594.000000
mean      537.407362
std       219.482257
min        91.030313
25%       380.416380
50%       552.477041
75%       693.231533
max      1284.225025
Name: corn_area_km2, dtype: float64


,county_fips,year,corn_area_km2
0,19001,2019,482.932853
1,19001,2020,470.617045
2,19001,2021,436.880502
3,19001,2022,435.481235
4,19001,2023,432.559493


In [ ]:
import os
import ee
import pandas as pd

# Define our three key temporal windows for feature engineering
WINDOWS = {
    'Jun': ('06-01', '06-30'),
    'Jul': ('07-01', '07-31'),
    'Aug': ('08-01', '08-31')
}

YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
S2_OUTPUT_FILE = f"{PROCESSED_DIR}/sentinel2_iowa_features_2019_2024.csv"

# Cloud-masking function
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    # Divide spectral bands by 10000 to obtain reflectance in [0, 1]
    return image.select(['B4', 'B5', 'B8', 'B11']).updateMask(mask).divide(10000)

# Compute indices
def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
    ndwi = image.normalizedDifference(['B8', 'B11']).rename('NDWI')
    return image.addBands([ndvi, ndre, ndwi])

yearly_files = []

for year in YEARS:
    year_cache_file = f"{PROCESSED_DIR}/s2_iowa_{year}_temp.csv"

    # Skip if this year was already computed and saved
    if os.path.exists(year_cache_file):
        print(f"Year {year} already cached. Loading from disk.")
        yearly_files.append(pd.read_csv(year_cache_file, dtype={'county_fips': str}))
        continue

    print(f"\n--- Extracting S2 Features for {year} ---")

    # Load annual CDL corn mask (class code 1)
    cdl = ee.ImageCollection('USDA/NASS/CDL').filterDate(f'{year}-01-01', f'{year}-12-31').first()
    corn_mask = cdl.select('cropland').eq(1)

    year_data = {}

    for month, (start_day, end_day) in WINDOWS.items():
        print(f"  Aggregating {month} imagery...")
        start_date = f"{year}-{start_day}"
        end_date = f"{year}-{end_day}"

        # Load collection, selecting only the necessary bands to reduce data transfer
        s2_collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
            .filterBounds(iowa_counties_ee) \
            .filterDate(start_date, end_date) \
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30)) \
            .select(['B4', 'B5', 'B8', 'B11', 'QA60']) \
            .map(mask_s2_clouds) \
            .map(add_indices)

        # Composite and apply corn mask
        s2_median = s2_collection.median().updateMask(corn_mask)
        bands_to_extract = s2_median.select(['NDVI', 'NDRE', 'NDWI'])

        # Aggregate at scale=100 for high computational throughput without memory errors
        stats = bands_to_extract.reduceRegions(
            collection=iowa_counties_ee,
            reducer=ee.Reducer.mean(),
            scale=100,
            tileScale=4
        )

        month_features = stats.select(['GEOID', 'NDVI', 'NDRE', 'NDWI']).getInfo()

        for feat in month_features['features']:
            fips = feat['properties']['GEOID']
            if fips not in year_data:
                year_data[fips] = {'county_fips': str(fips).zfill(5), 'year': year}

            year_data[fips][f'NDVI_mean_{month}'] = feat['properties'].get('NDVI')
            year_data[fips][f'NDRE_mean_{month}'] = feat['properties'].get('NDRE')
            year_data[fips][f'NDWI_mean_{month}'] = feat['properties'].get('NDWI')

    df_single_year = pd.DataFrame(list(year_data.values()))

    # Save checkpoint to disk
    df_single_year.to_csv(year_cache_file, index=False)
    yearly_files.append(df_single_year)
    print(f"  Successfully cached {year} data.")

# Combine all years
s2_df = pd.concat(yearly_files, ignore_index=True)

# Engineer temporal change features
print("\nComputing temporal change features...")
s2_df['NDVI_Jul_minus_Jun'] = s2_df['NDVI_mean_Jul'] - s2_df['NDVI_mean_Jun']
s2_df['NDVI_Aug_minus_Jul'] = s2_df['NDVI_mean_Aug'] - s2_df['NDVI_mean_Jul']
s2_df['NDRE_Jul_minus_Jun'] = s2_df['NDRE_mean_Jul'] - s2_df['NDRE_mean_Jun']

# Sort and write master satellite table
s2_df = s2_df.sort_values(["county_fips", "year"]).reset_index(drop=True)
s2_df.to_csv(S2_OUTPUT_FILE, index=False)

print(f"\nSentinel-2 extraction complete! Final table saved to: {S2_OUTPUT_FILE}")
display(s2_df.head())


--- Extracting S2 Features for 2019 ---
  Aggregating Jun imagery...
  Aggregating Jul imagery...
  Aggregating Aug imagery...
  Successfully cached 2019 data.

--- Extracting S2 Features for 2020 ---
  Aggregating Jun imagery...
  Aggregating Jul imagery...
  Aggregating Aug imagery...
  Successfully cached 2020 data.

--- Extracting S2 Features for 2021 ---
  Aggregating Jun imagery...
  Aggregating Jul imagery...
  Aggregating Aug imagery...
  Successfully cached 2021 data.

--- Extracting S2 Features for 2022 ---
  Aggregating Jun imagery...
  Aggregating Jul imagery...
  Aggregating Aug imagery...
  Successfully cached 2022 data.

--- Extracting S2 Features for 2023 ---
  Aggregating Jun imagery...
  Aggregating Jul imagery...
  Aggregating Aug imagery...
  Successfully cached 2023 data.

--- Extracting S2 Features for 2024 ---
  Aggregating Jun imagery...
  Aggregating Jul imagery...
  Aggregating Aug imagery...
  Successfully cached 2024 data.

Computing temporal change feature

,county_fips,year,NDVI_mean_Jun,NDRE_mean_Jun,NDWI_mean_Jun,NDVI_mean_Jul,NDRE_mean_Jul,NDWI_mean_Jul,NDVI_mean_Aug,NDRE_mean_Aug,NDWI_mean_Aug,NDVI_Jul_minus_Jun,NDVI_Aug_minus_Jul,NDRE_Jul_minus_Jun
0,19001,2019,0.484932,0.341685,-0.024885,0.824008,0.672907,0.354358,0.844296,0.679956,0.389890,0.339075,0.020288,0.331221
1,19001,2020,0.447023,0.308443,-0.098127,0.841672,0.702798,0.383632,0.792570,0.636177,0.330602,0.394649,-0.049102,0.394355
2,19001,2021,0.505986,0.370171,-0.048529,0.812703,0.677873,0.400456,0.814484,0.673281,0.386275,0.306717,0.001781,0.307702
3,19001,2022,0.479491,0.336320,-0.072827,0.801379,0.657544,0.317637,0.758613,0.609885,0.315651,0.321888,-0.042766,0.321224
4,19001,2023,0.526391,0.396109,0.026803,0.832832,0.695047,0.395064,0.825789,0.681231,0.376899,0.306440,-0.007042,0.298939


In [ ]:


# Define the months to cover early season through late pre-harvest
MONTHS = {
    5: 'May',
    6: 'Jun',
    7: 'Jul',
    8: 'Aug'
}

YEARS = [2019, 2020, 2021, 2022, 2023, 2024]
era5_results = []

print("Extracting ERA5-Land environmental features...")

for year in YEARS:
    print(f"Processing ERA5-Land for {year}...")

    year_data = {}

    for month_num, month_name in MONTHS.items():
        # Load the monthly aggregate ERA5-Land image
        era5_img = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR') \
            .filter(ee.Filter.date(f'{year}-{month_num:02d}-01', f'{year}-{month_num:02d}-28')) \
            .first()

        # Select target variables and convert units
        # Temperature: Kelvin to Celsius
        temp = era5_img.select('temperature_2m').subtract(273.15).rename('temp')

        # Precipitation: Meters to Millimeters
        precip = era5_img.select('total_precipitation_sum').multiply(1000).rename('precip')

        # Soil water (Layer 1, 0-7cm): m³/m³
        soil = era5_img.select('volumetric_soil_water_layer_1').rename('soil_water')

        # Combine into a single image
        weather_bands = ee.Image([temp, precip, soil])

        # Aggregate by county at ~11km scale
        stats = weather_bands.reduceRegions(
            collection=iowa_counties_ee,
            reducer=ee.Reducer.mean(),
            scale=11132,
            tileScale=1
        )

        # Transfer data to Colab
        month_features = stats.select(['GEOID', 'temp', 'precip', 'soil_water']).getInfo()

        for feat in month_features['features']:
            fips = feat['properties']['GEOID']
            if fips not in year_data:
                # Store FIPS as a 5-digit string to match previous tables
                year_data[fips] = {'county_fips': str(fips).zfill(5), 'year': year}

            # Store values with descriptive column names (e.g., 'temp_Jul')
            year_data[fips][f'temp_{month_name}'] = feat['properties'].get('temp')
            year_data[fips][f'precip_{month_name}'] = feat['properties'].get('precip')
            year_data[fips][f'soil_water_{month_name}'] = feat['properties'].get('soil_water')

    # Append this year's fully merged dictionary
    era5_results.extend(list(year_data.values()))

# Convert to DataFrame
era5_df = pd.DataFrame(era5_results)

# Sort dataset
era5_df = era5_df.sort_values(["county_fips", "year"]).reset_index(drop=True)

# Save the finalized environmental feature table
ERA5_OUTPUT_FILE = f"{PROCESSED_DIR}/era5_iowa_features_2019_2024.csv"
era5_df.to_csv(ERA5_OUTPUT_FILE, index=False)

print(f"\nEnvironmental extraction complete! Saved to {ERA5_OUTPUT_FILE}")
display(era5_df.head())

Extracting ERA5-Land environmental features...
Processing ERA5-Land for 2019...
Processing ERA5-Land for 2020...
Processing ERA5-Land for 2021...
Processing ERA5-Land for 2022...


Processing ERA5-Land for 2023...
Processing ERA5-Land for 2024...

Environmental extraction complete! Saved to /content/drive/MyDrive/Project_2_Crop_Yield/data_processed/era5_iowa_features_2019_2024.csv


,county_fips,year,temp_May,precip_May,soil_water_May,temp_Jun,precip_Jun,soil_water_Jun,temp_Jul,precip_Jul,soil_water_Jul,temp_Aug,precip_Aug,soil_water_Aug
0,19001,2019,14.686884,274.193878,0.391635,21.164972,119.763445,0.344689,24.907974,48.828074,0.237706,22.641081,109.877914,0.260988
1,19001,2020,14.894267,126.511769,0.339652,23.380038,178.218755,0.338046,24.554657,113.049079,0.312903,23.823271,14.840513,0.207871
2,19001,2021,15.590123,151.936273,0.351055,24.286236,59.009431,0.254435,24.345435,120.784056,0.251389,24.702661,98.720605,0.239974
3,19001,2022,16.745337,174.014040,0.379342,22.943697,95.731064,0.317264,24.592586,82.133833,0.263444,24.101330,82.129142,0.227163
4,19001,2023,18.138575,77.622774,0.309431,23.229731,55.056915,0.226479,23.943724,55.867079,0.212455,23.824545,87.537362,0.244550


# Master Data SET/
Add all the data

In [ ]:
import pandas as pd
import os

# Define file paths (assuming PROJECT_DIR is still active in your environment)
PROCESSED_DIR = f"{PROJECT_DIR}/data_processed"
CLEAN_YIELD_FILE = f"{PROCESSED_DIR}/nass_iowa_corn_yield_2019_2024_clean.csv"
CDL_OUTPUT_FILE = f"{PROCESSED_DIR}/cdl_iowa_corn_area_2019_2024.csv"
S2_OUTPUT_FILE = f"{PROCESSED_DIR}/sentinel2_iowa_features_2019_2024.csv"
ERA5_OUTPUT_FILE = f"{PROCESSED_DIR}/era5_iowa_features_2019_2024.csv"

# 1. Load the four pillars (Ensuring county_fips is loaded as a 5-digit string)
df_yield = pd.read_csv(CLEAN_YIELD_FILE, dtype={'county_fips': str})
df_cdl = pd.read_csv(CDL_OUTPUT_FILE, dtype={'county_fips': str})
df_s2 = pd.read_csv(S2_OUTPUT_FILE, dtype={'county_fips': str})
df_era5 = pd.read_csv(ERA5_OUTPUT_FILE, dtype={'county_fips': str})

# ==============================================================================
# BUILD THE MASTER DATASET
# ==============================================================================
print("Merging data sources...")

# Start with the NASS yield table (our supervised learning targets)
master_df = df_yield.merge(df_cdl, on=['county_fips', 'year'], how='left')
# Merge satellite features
master_df = master_df.merge(df_s2, on=['county_fips', 'year'], how='left')
# Merge environmental features
master_df = master_df.merge(df_era5, on=['county_fips', 'year'], how='left')

# Save Master Dataset immediately
MASTER_OUTPUT_FILE = f"{PROCESSED_DIR}/master_iowa_corn_yield_2019_2024.csv"
master_df.to_csv(MASTER_OUTPUT_FILE, index=False)
print(f"Master Dataset saved to: {MASTER_OUTPUT_FILE}")



Merging data sources...
Master Dataset saved to: /content/drive/MyDrive/Project_2_Crop_Yield/data_processed/master_iowa_corn_yield_2019_2024.csv


# LEAKAGE AUDIT

In [ ]:

print("\n================ STEP 9: DATA QUALITY & LEAKAGE AUDIT ================")

# A. Row/Key Integrity Check
duplicates = master_df.duplicated(subset=['county_fips', 'year']).sum()
print(f"\n1. Duplicate county-year rows: {duplicates} (Expected: 0)")
print(f"   Total rows: {len(master_df)} (Should match your cleaned NASS target table length)")

# B. Temporal Leakage Check: Forecast Cutoff Verification
# We must ensure no features are explicitly named September, October, or harvest-time.
columns = master_df.columns.tolist()
future_leaks = [col for col in columns if 'Sep' in col or 'Oct' in col or 'Nov' in col or 'Dec' in col]
print(f"\n2. Future-information leakage (post-August columns): {len(future_leaks)}")
if future_leaks:
    print(f"   WARNING - Leakage found in: {future_leaks}")
else:
    print("   SUCCESS: All temporal features honor the late-August pre-harvest forecast cutoff.")

# C. Missing Values Check
print("\n3. Missing Values Summary (Top 10 columns with missingness):")
missing_counts = master_df.isna().sum()
missing_vars = missing_counts[missing_counts > 0].sort_values(ascending=False)

if not missing_vars.empty:
    print(missing_vars.head(10))
    print("\n[SCIENTIFIC NOTE]: Missing Sentinel-2 values (NaNs) are normal and represent counties")
    print("where persistent cloud cover prevented a valid monthly composite. We retain these NaNs in")
    print("the master file. We will apply median-imputation later, strictly calculated from the")
    print("TRAINING years (2019-2022) to prevent forward data leakage into the test set.")
else:
    print("No missing values found!")

# D. Temporal Window Audit
years_present = sorted(master_df['year'].unique())
print(f"\n4. Temporal windows verified. Years present: {years_present}")

print("\n======================================================================")
print(f"Final Data Shape: {master_df.shape[0]} rows, {master_df.shape[1]} columns")

# 5. Display a final preview for visual inspection
display(master_df.head())


================ STEP 9: DATA QUALITY & LEAKAGE AUDIT ================

1. Duplicate county-year rows: 0 (Expected: 0)
   Total rows: 524 (Should match your cleaned NASS target table length)

2. Future-information leakage (post-August columns): 0
   SUCCESS: All temporal features honor the late-August pre-harvest forecast cutoff.

3. Missing Values Summary (Top 10 columns with missingness):
NDVI_mean_Jun         13
NDRE_mean_Jun         13
NDWI_mean_Jun         13
NDRE_Jul_minus_Jun    13
NDVI_Jul_minus_Jun    13
NDRE_mean_Aug          7
NDVI_mean_Aug          7
NDWI_mean_Aug          7
NDVI_Aug_minus_Jul     7
dtype: int64

[SCIENTIFIC NOTE]: Missing Sentinel-2 values (NaNs) are normal and represent counties
where persistent cloud cover prevented a valid monthly composite. We retain these NaNs in
the master file. We will apply median-imputation later, strictly calculated from the
TRAINING years (2019-2022) to prevent forward data leakage into the test set.

4. Temporal windows verifi

,county_fips,state,county,year,yield_bu_ac,corn_area_km2,NDVI_mean_Jun,NDRE_mean_Jun,NDWI_mean_Jun,NDVI_mean_Jul,...,soil_water_May,temp_Jun,precip_Jun,soil_water_Jun,temp_Jul,precip_Jul,soil_water_Jul,temp_Aug,precip_Aug,soil_water_Aug
0,19001,IOWA,ADAIR,2019,176.5,482.932853,0.484932,0.341685,-0.024885,0.824008,...,0.391635,21.164972,119.763445,0.344689,24.907974,48.828074,0.237706,22.641081,109.877914,0.260988
1,19001,IOWA,ADAIR,2020,168.7,470.617045,0.447023,0.308443,-0.098127,0.841672,...,0.339652,23.380038,178.218755,0.338046,24.554657,113.049079,0.312903,23.823271,14.840513,0.207871
2,19001,IOWA,ADAIR,2022,164.2,435.481235,0.479491,0.336320,-0.072827,0.801379,...,0.379342,22.943697,95.731064,0.317264,24.592586,82.133833,0.263444,24.101330,82.129142,0.227163
3,19001,IOWA,ADAIR,2024,211.2,437.384481,0.603641,0.463969,0.068008,0.818727,...,0.378069,23.445964,181.348044,0.326504,23.445967,73.906710,0.297937,23.147242,84.407761,0.245019
4,19003,IOWA,ADAMS,2020,173.3,297.958532,0.462265,0.318272,-0.096630,0.834350,...,0.345246,23.743920,158.442511,0.339056,24.756822,155.599440,0.324134,23.730057,16.030875,0.230045
